In [1]:
# Instalar PySpark
!pip install pyspark

In [2]:
# Importar SparkSession y otras librerías necesarias
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import rand, col, when
import pandas as pd
import numpy as np

# Crear sesión Spark
spark = SparkSession.builder \
    .appName("RegresionConMLlib") \
    .getOrCreate()

print("Spark sesión iniciada correctamente")

Spark sesión iniciada correctamente


In [3]:
# Número de registros
num_rows = 10000

# Crear un DataFrame de Spark con datos aleatorios
data = spark.range(num_rows).select(
    (rand() * 200 + 50).alias("superficie"),        # 50-250 m²
    (rand() * 5 + 1).cast("int").alias("habitaciones"),  # 1-6 habitaciones
    (rand() * 50).cast("int").alias("antiguedad"),      # 0-50 años
    (rand() * 4).cast("int").alias("ciudad_id")         # 0-3 (Madrid, Barcelona, Valencia, Sevilla)
)

# Asignar nombres de ciudades según el ID
data = data.withColumn("ciudad",
    when(col("ciudad_id") == 0, "Madrid")
    .when(col("ciudad_id") == 1, "Barcelona")
    .when(col("ciudad_id") == 2, "Valencia")
    .otherwise("Sevi lla")
)

# Crear variable objetivo: precio (fórmula simulada)
# Precio = 1000 * superficie + 20000 * habitaciones - 500 * antiguedad + ajuste_ciudad + ruido
data = data.withColumn("precio",
    1000 * col("superficie") +
    20000 * col("habitaciones") -
    500 * col("antiguedad") +
    when(col("ciudad") == "Madrid", 50000)
    .when(col("ciudad") == "Barcelona", 45000)
    .when(col("ciudad") == "Valencia", 30000)
    .otherwise(20000) +
    (rand() * 10000)  # ruido
)

# Seleccionar columnas finales
data = data.select("superficie", "habitaciones", "antiguedad", "ciudad", "precio")

# Mostrar algunos ejemplos
data.show(5, truncate=False)

+------------------+------------+----------+---------+------------------+
|superficie        |habitaciones|antiguedad|ciudad   |precio            |
+------------------+------------+----------+---------+------------------+
|104.32350357780939|5           |4         |Barcelona|253167.2665427865 |
|110.81983851078519|2           |9         |Barcelona|191411.65027646098|
|94.72594228966496 |1           |4         |Barcelona|165974.7809849267 |
|92.459705963078   |1           |22        |Valencia |135041.73061532338|
|229.83114014658403|3           |25        |Sevi lla |306364.0327847102 |
+------------------+------------+----------+---------+------------------+
only showing top 5 rows


## 3. Preparación de features
Vamos a construir un pipeline que:

1. Convierta la variable categórica "ciudad" en índices numéricos (StringIndexer).

2. Ensamble todas las features en un vector (VectorAssembler).

3. Normalice las features (opcional, pero recomendable para regresión lineal) con StandardScaler.

In [4]:
# Definir las etapas del pipeline de features
# 1. Indexar ciudad
city_indexer = StringIndexer(inputCol="ciudad", outputCol="ciudad_index")

# 2. Vector assembler: combinar todas las features
feature_cols = ["superficie", "habitaciones", "antiguedad", "ciudad_index"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")

# 3. Escalar features (normalización)
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

# Pipeline de preprocesamiento
preprocessing_pipeline = Pipeline(stages=[city_indexer, assembler, scaler])

# Aplicar el pipeline a los datos
prepared_data = preprocessing_pipeline.fit(data).transform(data)

# Mostrar el resultado
prepared_data.select("features", "precio").show(5, truncate=False)

+----------------------------------------------------------------------------------+------------------+
|features                                                                          |precio            |
+----------------------------------------------------------------------------------+------------------+
|[-0.7881092421422627,1.4013599072274312,-1.429225568985758,1.3568003142067155]    |253167.2665427865 |
|[-0.6756833311151179,-0.7207337385664703,-1.0816089889089597,1.3568003142067155]  |191411.65027646098|
|[-0.9542051381499255,-1.4280982871644374,-1.429225568985758,1.3568003142067155]   |165974.7809849267 |
|[-0.9934247421648065,-1.4280982871644374,-0.17780588070928371,-1.3283378496696256]|135041.73061532338|
|[1.383932459476826,-0.01336918996850309,0.030764067336795358,-0.43329179504417853]|306364.0327847102 |
+----------------------------------------------------------------------------------+------------------+
only showing top 5 rows


## 4. División en entrenamiento y prueba
Separamos los datos en 80% para entrenamiento y 20% para prueba.

In [6]:
train_data, test_data = prepared_data.randomSplit([0.8, 0.2], seed=42)
print(f"Tamaño entrenamiento: {train_data.count()} registros")
print(f"Tamaño prueba: {test_data.count()} registros")

Tamaño entrenamiento: 8064 registros
Tamaño prueba: 1936 registros


## 5. Entrenamiento del modelo de regresión
Usaremos regresión lineal con Spark MLlib.

In [7]:
# Crear el estimador de regresión lineal
lr = LinearRegression(featuresCol="features", labelCol="precio")

# Entrenar el modelo
lr_model = lr.fit(train_data)

# Mostrar coeficientes e intercepto
print("Coeficientes:", lr_model.coefficients)
print("Intercepto:", lr_model.intercept)

Coeficientes: [57825.05508726365,28312.70052770332,-7363.012545291836,8334.354865502244]
Intercepto: 239196.46485177358


## 6. Predicciones y evaluación
Generamos predicciones sobre los datos de prueba y evaluamos el modelo con métricas comunes: RMSE, MAE y R2.

In [8]:
# Predicciones
predictions = lr_model.transform(test_data)

# Mostrar algunas predicciones
predictions.select("precio", "prediction").show(10, truncate=False)

+------------------+------------------+
|precio            |prediction        |
+------------------+------------------+
|114426.60596232122|127433.10063666601|
|160631.40722837715|160598.46547800704|
|178197.35287330896|171943.37292737275|
|161182.0940944916 |161141.42780802533|
|116609.85942196171|127519.41249926786|
|165550.7613200822 |160304.887257617  |
|176483.2734280724 |171345.20166742383|
|164997.08523133432|158806.8041054781 |
|170500.2230969533 |186509.6303365765 |
|183162.5834812193 |174454.59242814593|
+------------------+------------------+
only showing top 10 rows


In [9]:
# Evaluador
evaluator_rmse = RegressionEvaluator(labelCol="precio", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="precio", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="precio", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.4f}")

RMSE: 8913.39
MAE: 7555.78
R2: 0.9814


## 7. Uso del pipeline completo
Una ventaja de Spark MLlib es que podemos encapsular todo el proceso (preprocesamiento + modelo) en un único Pipeline, lo que facilita su reutilización y despliegue.

In [11]:
# Construir pipeline completo con todas las etapas
full_pipeline = Pipeline(stages=[
    city_indexer,
    assembler,
    scaler,
    lr
])

# Entrenar el pipeline con los datos originales (sin transformar)
# Nota: Necesitamos los datos originales (con ciudad como texto) para el pipeline
data_original = data  # el DataFrame original con columnas sin transformar

# Dividir originales
train_orig, test_orig = data_original.randomSplit([0.8, 0.2], seed=42)

# Entrenar pipeline
pipeline_model = full_pipeline.fit(train_orig)

# Predecir con pipeline
predictions_pipeline = pipeline_model.transform(test_orig)

# Evaluar igual que antes
rmse_pipe = evaluator_rmse.evaluate(predictions_pipeline)
mae_pipe = evaluator_mae.evaluate(predictions_pipeline)
r2_pipe = evaluator_r2.evaluate(predictions_pipeline)

print("\n--- Resultados con Pipeline completo ---")
print(f"RMSE: {rmse_pipe:.2f}")
print(f"MAE: {mae_pipe:.2f}")
print(f"R2: {r2_pipe:.4f}")


--- Resultados con Pipeline completo ---
RMSE: 12185.68
MAE: 10629.05
R2: 0.9652


## 8. Guardar y cargar el modelo (opcional)
Spark permite guardar modelos entrenados para usarlos posteriormente.

In [12]:
# Guardar el modelo del pipeline
pipeline_model.write().overwrite().save("modelo_prediccion_precios")

# Cargarlo
from pyspark.ml import PipelineModel
loaded_model = PipelineModel.load("modelo_prediccion_precios")

## Conclusión
Hemos implementado un flujo completo de regresión con Spark MLlib en Colab, incluyendo:

* Creación de datos sintéticos.

* Preprocesamiento con StringIndexer, VectorAssembler y StandardScaler.

* Entrenamiento de regresión lineal.

* Evaluación con RMSE, MAE y R2.

* Uso de un Pipeline para unificar el proceso.

Spark MLlib permite escalar este tipo de flujos a datasets masivos distribuidos, manteniendo una sintaxis similar a scikit-learn pero adaptada a entornos Big Data.